# Rioxarray

## 概述

[`rioxarray`](https://corteva.github.io/rioxarray/)是强大的Python库Xarray的扩展，专注于地理空间栅格数据。它使用Xarray的标记多维数据结构提供对地理参考信息和地理空间变换的便捷访问，使其成为处理卫星图像或气候数据等地理空间数据的理想工具。

`rioxarray`的关键特性是其将rasterio的地理空间数据处理能力（如CRS和仿射变换）与Xarray的高效多维数组处理无缝集成。这使您能够轻松操作、分析和可视化栅格数据。

## 学习目标

在本讲座结束时，您应该能够：
- 了解`rioxarray`如何扩展Xarray以处理地理空间数据。
- 使用`rioxarray`加载和检查地理参考栅格数据集。
- 使用`rioxarray`执行基本的地理空间操作，如裁剪、重投影和掩膜。
- 使用`rioxarray`管理栅格数据集中的CRS和空间维度。
- Export and visualize geospatial raster datasets.

## 安装 rioxarray
若要使用 `rioxarray`，你需要将其与 `rasterio` 及其依赖项一同安装。你可以通过 pip 进行安装，方法是取消注释以下代码行：



In [ ]:
# %pip install rioxarray rasterio

## 导入rioxarray

您可以从导入`rioxarray`和其他必要的库开始：

In [ ]:
import rioxarray
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

xr.set_options(keep_attrs=True, display_expand_data=False)

## 加载地理参考栅格数据

`rioxarray`的核心功能之一是能够加载地理参考栅格数据，包括其CRS和地理空间变换。您可以使用`rioxarray`直接加载栅格文件（例如GeoTIFF文件）：

In [ ]:
# Load a raster dataset using rioxarray
url = "https://github.com/opengeos/datasets/releases/download/raster/LC09_039035_20240708_90m.tif"
data = rioxarray.open_rasterio(url)
data

在这里，`rioxarray.open_rasterio`将栅格数据加载到Xarray `DataArray`中，并自动附加地理空间元数据，包括CRS、仿射变换和空间坐标。

### 检查数据集

您可以轻松检查加载的数据集，包括其维度、坐标和属性：

In [ ]:
# View the structure of the DataArray
data.dims  # Dimensions (e.g., band, y, x)

In [ ]:
data.coords  # Coordinates (e.g., y, x in geographic or projected CRS)

In [ ]:
print(data.attrs)  # Metadata (including CRS)

### 检查CRS和变换信息

`rioxarray`将CRS和仿射变换元数据集成到Xarray对象中：

In [ ]:
# Check the CRS of the dataset
data.rio.crs

In [ ]:
# Check the affine transformation (mapping pixel coordinates to geographic coordinates)
data.rio.transform()

有时，栅格数据可能没有CRS，或者CRS可能不正确。如有必要，您可以手动分配CRS：

In [ ]:
# If the CRS is missing or incorrect, assign a CRS
data = data.rio.write_crs("EPSG:32611", inplace=True)

## 基本地理空间操作

### 重投影数据集

在地理空间分析中，将栅格数据重投影到另一个CRS是很常见的。例如，您可能希望将数据集从其原生投影重投影到WGS84地理坐标系（EPSG:4326）：

In [ ]:
# Reproject the dataset to WGS84 (EPSG:4326)
data_reprojected = data.rio.reproject("EPSG:4326")
print(data_reprojected.rio.crs)

### 裁剪栅格

当您只想关注特定地理区域时，裁剪栅格数据集非常有用。您可以使用与数据相同CRS的边界框来裁剪数据集：

In [ ]:
# Define a bounding box (in the same CRS as the dataset)
bbox = [-115.391, 35.982, -114.988, 36.425]

# Clip the raster to the bounding box
clipped_data = data_reprojected.rio.clip_box(*bbox)

In [ ]:
clipped_data.shape

或者，您可以使用包含多边形几何的矢量数据集来裁剪栅格：

In [ ]:
import geopandas as gpd

# Load a geojson with regions of interest
geojson_path = "https://github.com/opengeos/datasets/releases/download/places/las_vegas_bounds_utm.geojson"
bounds = gpd.read_file(geojson_path)

# Clip the raster to the shape
clipped_data2 = data.rio.clip(bounds.geometry, bounds.crs)

In [ ]:
clipped_data2.shape

## 处理空间维度

`rioxarray`支持对空间维度（纬度/经度或x/y坐标）进行操作，如重采样、缩减或切片。

### 重采样

要将栅格数据集重采样到不同的分辨率（例如1公里），使用`rio.resample`方法：

In [ ]:
# Resample to 1km resolution (using an average resampling method)
resampled_data = data.rio.reproject(data.rio.crs, resolution=(1000, 1000))

In [ ]:
resampled_data.shape

### 提取空间子集

您可以通过选择特定坐标范围来提取数据集的空间子集：

In [ ]:
# Select a subset of the data within a lat/lon range
min_x, max_x = -115.391, -114.988
min_y, max_y = 35.982, 36.425
subset = data_reprojected.sel(
    x=slice(min_x, max_x), y=slice(max_y, min_y)
)  # Slice y in reverse order
subset.shape

## 地理参考数据的可视化

对数据执行操作后，您可以使用matplotlib进行可视化。例如，使用波段4、3和2绘制多波段图像：

In [ ]:
# Plot the raster data
plt.figure(figsize=(8, 8))
data_reprojected.sel(band=[4, 3, 2]).plot.imshow(vmin=0, vmax=0.3)
plt.title("Landsat Image covering Las Vegas")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

您也可以用同样的方式可视化裁剪或掩膜的数据：

In [ ]:
# Plot the raster data
plt.figure(figsize=(8, 8))
clipped_data.sel(band=[4, 3, 2]).plot.imshow(vmin=0, vmax=0.3)
plt.title("Landsat Image covering Las Vegas")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

对于更高级的绘图，例如在栅格数据上叠加矢量数据集，您可以将`rioxarray`与`geopandas`和`matplotlib`结合使用：

In [ ]:
# Plot raster with GeoJSON overlay
fig, ax = plt.subplots(figsize=(8, 8))
data.attrs["long_name"] = "Surface Reflectance"  # Update the long_name attribute
data.sel(band=4).plot.imshow(ax=ax, vmin=0, vmax=0.4, cmap="gray")
bounds.boundary.plot(ax=ax, color="red")
plt.title("Raster with Vector Overlay")
plt.show()

## 保存数据

与加载数据一样，您可以将`rioxarray`数据集导出到磁盘。例如，您可以将修改或处理后的栅格数据保存为GeoTIFF文件：

In [ ]:
# Save the DataArray as a GeoTIFF file
data.rio.to_raster("output_raster.tif")

## 处理NoData值

如果您的数据集包含NoData值，您可以使用以下函数进行管理：

In [ ]:
# Assign NoData value
data2 = data.rio.set_nodata(-9999)

# Remove NoData values (mask them)
data_clean = data2.rio.write_nodata(-9999, inplace=True)

## 重投影到多个CRS

您可以将数据集重投影到多个CRS并进行比较。例如：

In [ ]:
# Reproject to WGS 84 (EPSG:4326)
data = data.rio.reproject("EPSG:4326")
print(data.rio.crs)

In [ ]:
# Reproject to EPSG:3857 (Web Mercator)
mercator_data = data.rio.reproject("EPSG:3857")
print(mercator_data.rio.crs)

In [ ]:
# Plot the raster data in WGS84
plt.figure(figsize=(6, 6))
data.sel(band=[4, 3, 2]).plot.imshow(vmin=0, vmax=0.3)
plt.title("EPSG:4326")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
# Plot the raster data in Web Mercator
plt.figure(figsize=(6, 6))
mercator_data.sel(band=[4, 3, 2]).plot.imshow(vmin=0, vmax=0.3)
plt.title("EPSG:3857")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()

## 基本波段运算（NDVI计算）

波段运算使我们能够跨不同波段执行计算。一个常见的应用是计算归一化植被指数（NDVI），这是植被健康状况的指标。

NDVI的计算公式为：

NDVI = (NIR - Red) / (NIR + Red)

我们可以按如下方式计算和绘制NDVI：

In [ ]:
# Select the red (band 4) and NIR (band 5) bands
red_band = data.sel(band=4)
nir_band = data.sel(band=5)

# Calculate NDVI
ndvi = (nir_band - red_band) / (nir_band + red_band)
ndvi = ndvi.clip(min=-1, max=1)  # Clip values to the range [-1, 1]
ndvi.attrs["long_name"] = "NDVI"

为了可视化NDVI，我们可以使用matplotlib绘制它：

In [ ]:
# Plot the NDVI values
ndvi.plot(cmap="RdYlGn", vmin=-1, vmax=1)
plt.title("NDVI of the Landsat Image")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

您还可以通过应用阈值来掩膜非植被区域或具有无效NDVI值的区域（如水体或城市区域）：

In [ ]:
# Mask out non-vegetated areas (NDVI < 0.2)
ndvi_clean = ndvi.where(ndvi > 0.2)
ndvi_clean.plot(cmap="Greens", vmin=0.2, vmax=0.5)
plt.title("Cleaned NDVI (non-vegetated areas masked)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## 练习

### 示例数据集

在练习中，我们将使用利比亚的示例GeoTIFF栅格数据集，可在以下URL获取：
https://github.com/opengeos/datasets/releases/download/raster/Libya-2023-09-13.tif

### 练习1：加载和检查栅格数据集

1. 使用`rioxarray`加载GeoTIFF栅格文件。
2. 通过打印数据集的维度、坐标和属性来检查数据集。
2. 通过打印数据集的维度、坐标和属性来检查数据集。
2. 通过打印数据集的维度、坐标和属性来检查数据集。
3. 检查并打印数据集的CRS和仿射变换。

### 练习2：将栅格重投影到新的CRS

1. 将加载的栅格数据集从其原始CRS重投影到EPSG:4326（WGS84）。
2. 打印新的CRS并检查重投影数据的维度和坐标。
3. 绘制原始和重投影数据集进行比较。
3. 绘制原始和重投影数据集进行比较。
3. 绘制原始和重投影数据集进行比较。

### 练习3：使用边界框裁剪栅格

1. 定义一个覆盖利比亚陆地面积的边界框（例如`xmin`、`ymin`、`xmax`、`ymax`）。
2. 使用此边界框裁剪栅格数据集。
3. 绘制裁剪后的数据以可视化结果。

### 练习4：使用矢量数据集掩膜栅格

1. 使用`geopandas`加载位于https://github.com/opengeos/datasets/releases/download/raster/Derna_Libya.geojson的GeoJSON文件。
2. 使用GeoJSON掩膜栅格数据集，只保留GeoJSON边界内的数据。
3. 绘制掩膜后的栅格数据。

### 练习5：将栅格重采样到不同分辨率

1. 使用平均重采样方法将栅格数据集重采样到3米分辨率。
2. 检查重采样后的新维度和坐标。
3. 将重采样后的栅格数据集保存为新的GeoTIFF文件。

## 总结

在本讲座中，我们探讨了`rioxarray`的基本功能，这是一个为地理空间栅格数据设计的强大Xarray扩展。要点包括：

- 加载和检查具有CRS和变换元数据的地理空间栅格数据。
- 执行基本的地理空间操作，如重投影、裁剪和掩膜。
- 可视化和导出栅格数据。
- 使用切片、重采样和其他操作处理空间维度（x, y）。

通过将Xarray的多维数据处理能力与rasterio的地理空间特性相结合，`rioxarray`使管理和分析地理空间栅格数据集变得更加容易。对于在科学计算、环境分析或遥感领域处理地理空间数据的任何人来说，它都是一个多功能的工具。